# 🛒 TechnoMart — SageMaker ML Training Notebook

Pipeline training lengkap untuk semua ML task TechnoMart.

| # | Task | Dataset | Model |
|---|------|---------|-------|
| 1 | Churn Prediction | user_profiles | XGBoost |
| 2 | User Segmentation | user_profiles | XGBoost Multi-class |
| 3 | LTV Prediction | user_profiles | LightGBM Regressor |
| 4 | Fraud Detection | transaction_history | XGBoost + SMOTE |
| 5 | Sentiment Analysis | product_reviews | LightGBM + TF-IDF |
| 6 | Rating Prediction | product_reviews | LightGBM Regressor |
| 7 | Product Trending | product_catalog | XGBoost |
| 8 | CTR / CVR Prediction | product_catalog | LightGBM Regressor |
| 9 | Search CTR | search_logs | LightGBM |
| 10 | Collaborative Filtering | user_interactions | Matrix Factorization (PyTorch) |
| 11 | Search Ranking | search_logs | LightGBM LambdaRank |

---
**Instance:** `ml.m5.4xlarge` (16 vCPU, 64GB RAM) — bisa disesuaikan  
**Kernel:** `Python 3 (Data Science 3.0)`

## 0. Setup & Instalasi

In [ ]:
# Install dependensi tambahan
!pip install -q lightgbm imbalanced-learn optuna scikit-learn shap torch torchvision --upgrade
!pip install -q sagemaker --upgrade

In [ ]:
import os
import json
import warnings
import logging
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sagemaker
import shap

from io import StringIO
from datetime import datetime

# ML Libraries
import xgboost as xgb
import lightgbm as lgb
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, ndcg_score
)
from imblearn.over_sampling import SMOTE

# SageMaker
from sagemaker import get_execution_role
from sagemaker.session import Session

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('✅ Library berhasil diimport')
print(f'   XGBoost  : {xgb.__version__}')
print(f'   LightGBM : {lgb.__version__}')
print(f'   PyTorch  : {torch.__version__}')
print(f'   CUDA     : {torch.cuda.is_available()}')

In [ ]:
# ── Konfigurasi AWS & SageMaker ──────────────────────────────────────────────
S3_BUCKET    = 'technomart-s3'
S3_RAW       = f's3://{S3_BUCKET}/raw-data'
S3_PROCESSED = f's3://{S3_BUCKET}/processed-data'
S3_MODELS    = f's3://{S3_BUCKET}/models'
AWS_REGION   = 'us-east-1'

session     = sagemaker.Session()
role        = get_execution_role()
s3_client   = boto3.client('s3', region_name=AWS_REGION)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'✅ SageMaker session ready')
print(f'   Role    : {role[:60]}...')
print(f'   Region  : {AWS_REGION}')
print(f'   Bucket  : {S3_BUCKET}')
print(f'   Device  : {DEVICE}')

## 1. Helper Functions

In [ ]:
# ── Baca CSV dari S3 ──────────────────────────────────────────────────────────
def read_s3_csv(s3_path: str, nrows: int = None) -> pd.DataFrame:
    """Baca CSV dari S3 path. Mendukung chunked files (part001, part002, dst)."""
    import re
    bucket, key = s3_path.replace('s3://', '').split('/', 1)
    
    # List objects di prefix
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=key.rstrip('/') + '/')
    csv_keys = [
        obj['Key'] for obj in resp.get('Contents', [])
        if obj['Key'].endswith('.csv')
    ]
    
    if not csv_keys:
        # Coba baca langsung sebagai file
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        return pd.read_csv(obj['Body'], nrows=nrows)
    
    dfs = []
    total = 0
    for k in sorted(csv_keys):
        obj  = s3_client.get_object(Bucket=bucket, Key=k)
        remaining = None if nrows is None else max(0, nrows - total)
        df   = pd.read_csv(obj['Body'], nrows=remaining)
        dfs.append(df)
        total += len(df)
        if nrows and total >= nrows:
            break
    
    result = pd.concat(dfs, ignore_index=True)
    logger.info(f'Read {len(result):,} rows from {s3_path}')
    return result


# ── Save model ke S3 ──────────────────────────────────────────────────────────
def save_model_to_s3(model_obj, model_name: str, metadata: dict = None):
    """Simpan model (pkl/pt) ke S3 dengan metadata."""
    import pickle
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    key = f'models/{model_name}/{model_name}_{timestamp}.pkl'
    
    buf = pickle.dumps(model_obj)
    s3_client.put_object(Bucket=S3_BUCKET, Key=key, Body=buf)
    
    if metadata:
        meta_key = key.replace('.pkl', '_metadata.json')
        s3_client.put_object(
            Bucket=S3_BUCKET,
            Key=meta_key,
            Body=json.dumps(metadata, indent=2, default=str)
        )
    logger.info(f'✅ Model saved: s3://{S3_BUCKET}/{key}')
    return f's3://{S3_BUCKET}/{key}'


# ── Evaluation report ─────────────────────────────────────────────────────────
def eval_classifier(y_true, y_pred, y_prob=None, label='Model'):
    print(f'\n📊 {label} — Classification Report')
    print('=' * 55)
    print(classification_report(y_true, y_pred))
    if y_prob is not None and len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_prob)
        print(f'   ROC-AUC : {auc:.4f}')
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'auc': roc_auc_score(y_true, y_prob) if y_prob is not None and len(np.unique(y_true)) == 2 else None
    }


def eval_regressor(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {label} — Regression Metrics')
    print('=' * 40)
    print(f'   RMSE : {rmse:>12,.2f}')
    print(f'   MAE  : {mae:>12,.2f}')
    print(f'   R²   : {r2:>12.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


# ── Plot feature importance ───────────────────────────────────────────────────
def plot_feature_importance(model, feature_names, top_n=20, title='Feature Importance'):
    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
    elif hasattr(model, 'feature_importance'):
        imp = model.feature_importance()
    else:
        return
    
    df_imp = pd.DataFrame({'feature': feature_names, 'importance': imp})
    df_imp = df_imp.nlargest(top_n, 'importance')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=df_imp, x='importance', y='feature', palette='viridis', ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()


print('✅ Helper functions ready')

## 2. Load Data dari S3

In [ ]:
# ── Load semua dataset ────────────────────────────────────────────────────────
# Gunakan nrows untuk development, hapus untuk full training
SAMPLE = None  # Set ke integer (misal 50_000) untuk quick dev

print('📥 Loading datasets dari S3...')

df_users    = read_s3_csv(f'{S3_RAW}/user-profiles',       nrows=SAMPLE)
df_products = read_s3_csv(f'{S3_RAW}/product-catalog',     nrows=SAMPLE)
df_interact = read_s3_csv(f'{S3_RAW}/user-interactions',   nrows=SAMPLE)
df_txn      = read_s3_csv(f'{S3_RAW}/transaction-history', nrows=SAMPLE)
df_reviews  = read_s3_csv(f'{S3_RAW}/product-reviews',     nrows=SAMPLE)
df_search   = read_s3_csv(f'{S3_RAW}/search-logs',         nrows=SAMPLE)

print('\n📋 Dataset Summary:')
for name, df in [
    ('user_profiles',       df_users),
    ('product_catalog',     df_products),
    ('user_interactions',   df_interact),
    ('transaction_history', df_txn),
    ('product_reviews',     df_reviews),
    ('search_logs',         df_search),
]:
    print(f'  {name:<25} : {len(df):>9,} rows × {len(df.columns):>3} cols')

## 3. Task 1: Churn Prediction (XGBoost Binary Classification)

In [ ]:
print('=' * 60)
print('TASK 1: Churn Prediction')
print('=' * 60)

# ── Feature engineering ───────────────────────────────────────────────────────
churn_features = [
    'age', 'tenure_days', 'loyalty_score', 'total_spend_idr',
    'avg_order_value_idr', 'total_orders', 'total_items_purchased',
    'days_since_last_purchase', 'email_open_rate', 'push_notif_enabled',
    'is_premium_member', 'referral_count', 'avg_session_duration_sec',
    'avg_pages_per_session'
]
cat_features_churn = ['gender', 'city', 'preferred_device', 'acquisition_channel']

df_churn = df_users[churn_features + cat_features_churn + ['is_churned']].copy()
df_churn = df_churn.dropna()

# Encode kategorik
le = LabelEncoder()
for col in cat_features_churn:
    df_churn[col] = le.fit_transform(df_churn[col].astype(str))

X_churn = df_churn.drop('is_churned', axis=1)
y_churn = df_churn['is_churned']

print(f'Churn rate : {y_churn.mean():.2%}')
print(f'Features   : {X_churn.shape[1]}')
print(f'Samples    : {len(X_churn):,}')

X_tr, X_te, y_tr, y_te = train_test_split(
    X_churn, y_churn, test_size=0.2, stratify=y_churn, random_state=RANDOM_SEED
)

In [ ]:
# ── Optuna Hyperparameter Tuning ──────────────────────────────────────────────
def objective_churn(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 1),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
        'scale_pos_weight': (y_tr == 0).sum() / (y_tr == 1).sum(),  # imbalance handling
        'use_label_encoder': False,
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_tr, y_tr, cv=3, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study_churn = optuna.create_study(direction='maximize', study_name='churn_xgb')
study_churn.optimize(objective_churn, n_trials=30, show_progress_bar=True)

best_params_churn = study_churn.best_params
best_params_churn.update({
    'scale_pos_weight': (y_tr == 0).sum() / (y_tr == 1).sum(),
    'use_label_encoder': False,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
})
print(f'\n✅ Best AUC (CV): {study_churn.best_value:.4f}')

In [ ]:
# ── Train final model ─────────────────────────────────────────────────────────
churn_model = xgb.XGBClassifier(**best_params_churn)
churn_model.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    verbose=100
)

y_pred_churn = churn_model.predict(X_te)
y_prob_churn = churn_model.predict_proba(X_te)[:, 1]

churn_metrics = eval_classifier(y_te, y_pred_churn, y_prob_churn, label='Churn XGBoost')
plot_feature_importance(churn_model, X_churn.columns.tolist(), title='Churn — Feature Importance')

# SHAP explanation
explainer = shap.TreeExplainer(churn_model)
shap_vals  = explainer.shap_values(X_te[:500])
shap.summary_plot(shap_vals, X_te[:500], feature_names=X_churn.columns.tolist(), show=True)

# Save model
churn_model_path = save_model_to_s3(
    churn_model, 'churn-prediction',
    metadata={'metrics': churn_metrics, 'best_params': best_params_churn, 'features': X_churn.columns.tolist()}
)

## 4. Task 2: User Segmentation (XGBoost Multi-class)

In [ ]:
print('=' * 60)
print('TASK 2: User Segmentation (bronze/silver/gold/platinum)')
print('=' * 60)

df_seg = df_users[churn_features + cat_features_churn + ['user_segment']].copy().dropna()

le_seg = LabelEncoder()
for col in cat_features_churn:
    df_seg[col] = le_seg.fit_transform(df_seg[col].astype(str))

le_target = LabelEncoder()
df_seg['segment_encoded'] = le_target.fit_transform(df_seg['user_segment'])
print('Segment mapping:', dict(enumerate(le_target.classes_)))
print(df_seg['user_segment'].value_counts())

X_seg = df_seg.drop(['user_segment', 'segment_encoded'], axis=1)
y_seg = df_seg['segment_encoded']

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_seg, y_seg, test_size=0.2, stratify=y_seg, random_state=RANDOM_SEED
)

seg_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
seg_model.fit(X_tr_s, y_tr_s, eval_set=[(X_te_s, y_te_s)], verbose=100)

y_pred_s = seg_model.predict(X_te_s)
seg_metrics = eval_classifier(y_te_s, y_pred_s, label='User Segmentation XGBoost')
plot_feature_importance(seg_model, X_seg.columns.tolist(), title='Segmentation — Feature Importance')

save_model_to_s3(seg_model, 'user-segmentation',
    metadata={'metrics': seg_metrics, 'classes': le_target.classes_.tolist()})

## 5. Task 3: LTV Prediction (LightGBM Regression)

In [ ]:
print('=' * 60)
print('TASK 3: Customer Lifetime Value Prediction')
print('=' * 60)

ltv_features = churn_features + ['loyalty_score', 'referral_count']
df_ltv = df_users[ltv_features + cat_features_churn + ['ltv_idr']].copy().dropna()

# Log-transform target (LTV skewed)
df_ltv['log_ltv'] = np.log1p(df_ltv['ltv_idr'])

for col in cat_features_churn:
    df_ltv[col] = LabelEncoder().fit_transform(df_ltv[col].astype(str))

X_ltv = df_ltv.drop(['ltv_idr', 'log_ltv'], axis=1)
y_ltv = df_ltv['log_ltv']  # train pada log-scale
y_ltv_orig = df_ltv['ltv_idr']

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_ltv, y_ltv, test_size=0.2, random_state=RANDOM_SEED
)

# Optuna tuning
def objective_ltv(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X_tr_l, y_tr_l, cv=3, scoring='neg_root_mean_squared_error')
    return scores.mean()

study_ltv = optuna.create_study(direction='maximize', study_name='ltv_lgbm')
study_ltv.optimize(objective_ltv, n_trials=30, show_progress_bar=True)

ltv_model = lgb.LGBMRegressor(**study_ltv.best_params, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)
ltv_model.fit(X_tr_l, y_tr_l, eval_set=[(X_te_l, y_te_l)])

# Inverse-transform prediksi ke IDR asli
y_pred_ltv_log = ltv_model.predict(X_te_l)
y_pred_ltv     = np.expm1(y_pred_ltv_log)
y_true_ltv     = np.expm1(y_te_l.values)

ltv_metrics = eval_regressor(y_true_ltv, y_pred_ltv, label='LTV LightGBM')
plot_feature_importance(ltv_model, X_ltv.columns.tolist(), title='LTV — Feature Importance')

save_model_to_s3(ltv_model, 'ltv-prediction', metadata={'metrics': ltv_metrics})

## 6. Task 4: Fraud Detection (XGBoost + SMOTE)

In [ ]:
print('=' * 60)
print('TASK 4: Fraud Detection')
print('=' * 60)

fraud_features = [
    'quantity', 'unit_price_idr', 'discount_amt_idr', 'shipping_fee_idr',
    'subtotal_idr', 'total_idr', 'hour_of_day', 'day_of_week', 'is_weekend',
    'delivery_days'
]
cat_fraud = ['payment_method', 'payment_status', 'platform']

df_fraud = df_txn[fraud_features + cat_fraud + ['is_fraud']].copy().dropna()
for col in cat_fraud:
    df_fraud[col] = LabelEncoder().fit_transform(df_fraud[col].astype(str))

X_fraud = df_fraud.drop('is_fraud', axis=1)
y_fraud = df_fraud['is_fraud']

print(f'Fraud rate : {y_fraud.mean():.3%}')

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=RANDOM_SEED
)

# SMOTE untuk handle extreme imbalance
smote = SMOTE(sampling_strategy=0.3, random_state=RANDOM_SEED)
X_tr_f_res, y_tr_f_res = smote.fit_resample(X_tr_f, y_tr_f)
print(f'After SMOTE — class counts: {dict(zip(*np.unique(y_tr_f_res, return_counts=True)))}')

fraud_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,  # SMOTE sudah handle imbalance
    eval_metric='aucpr',  # PR-AUC lebih informatif untuk fraud
    tree_method='hist',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
fraud_model.fit(
    X_tr_f_res, y_tr_f_res,
    eval_set=[(X_te_f, y_te_f)],
    verbose=100
)

y_pred_fraud = fraud_model.predict(X_te_f)
y_prob_fraud = fraud_model.predict_proba(X_te_f)[:, 1]
fraud_metrics = eval_classifier(y_te_f, y_pred_fraud, y_prob_fraud, label='Fraud XGBoost')
plot_feature_importance(fraud_model, X_fraud.columns.tolist(), title='Fraud — Feature Importance')

# Visualisasi confusion matrix
cm = confusion_matrix(y_te_f, y_pred_fraud)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Not Fraud','Fraud'], yticklabels=['Not Fraud','Fraud'])
ax.set_title('Fraud Detection — Confusion Matrix')
plt.tight_layout(); plt.show()

save_model_to_s3(fraud_model, 'fraud-detection', metadata={'metrics': fraud_metrics})

## 7. Task 5 & 6: Sentiment Analysis & Rating Prediction (LightGBM + TF-IDF)

In [ ]:
print('=' * 60)
print('TASK 5: Sentiment Analysis + TASK 6: Rating Prediction')
print('=' * 60)

df_rev = df_reviews.dropna(subset=['review_text', 'rating', 'sentiment'])

# TF-IDF features dari review text
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True
)
tfidf_matrix = tfidf.fit_transform(df_rev['review_text'].astype(str))
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# Numerik features
num_review_features = ['helpful_votes', 'verified_purchase', 'has_photo']
df_rev_num = df_rev[num_review_features].reset_index(drop=True)

# Gabung
X_rev = pd.concat([df_rev_num, tfidf_df], axis=1)

# ── Task 5: Sentiment ──────────────────────────────────────────────────────────
le_sent = LabelEncoder()
y_sent = le_sent.fit_transform(df_rev['sentiment'].reset_index(drop=True))
print('Sentiment mapping:', dict(enumerate(le_sent.classes_)))

X_tr_rv, X_te_rv, y_tr_s5, y_te_s5 = train_test_split(
    X_rev, y_sent, test_size=0.2, stratify=y_sent, random_state=RANDOM_SEED
)

sentiment_model = lgb.LGBMClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)
sentiment_model.fit(X_tr_rv, y_tr_s5)
y_pred_sent = sentiment_model.predict(X_te_rv)
sent_metrics = eval_classifier(y_te_s5, y_pred_sent, label='Sentiment LightGBM')

# ── Task 6: Rating Prediction ─────────────────────────────────────────────────
y_rating = df_rev['rating'].astype(int).reset_index(drop=True)
_, _, y_tr_r, y_te_r = train_test_split(X_rev, y_rating, test_size=0.2, random_state=RANDOM_SEED)

rating_model = lgb.LGBMRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    num_leaves=63, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
)
rating_model.fit(X_tr_rv, y_tr_r)
y_pred_r = np.clip(np.round(rating_model.predict(X_te_rv)), 1, 5)
rating_metrics = eval_regressor(y_te_r, y_pred_r, label='Rating LightGBM')

save_model_to_s3({'sentiment': sentiment_model, 'rating': rating_model, 'tfidf': tfidf},
                 'review-models',
                 metadata={'sentiment': sent_metrics, 'rating': rating_metrics})

## 8. Task 7, 8: Product Trending, CTR/CVR (LightGBM)

In [ ]:
print('=' * 60)
print('TASK 7: Product Trending + TASK 8: CTR/CVR Prediction')
print('=' * 60)

prod_features = [
    'base_price_idr', 'discount_pct', 'final_price_idr', 'weight_gram',
    'is_digital', 'is_imported', 'stock_quantity', 'avg_rating',
    'review_count', 'views_last_30d', 'sales_last_30d', 'days_since_listed'
]
cat_prod = ['category', 'subcategory', 'brand']

df_prod = df_products[prod_features + cat_prod + ['is_trending', 'click_through_rate', 'conversion_rate']].copy().dropna()
for col in cat_prod:
    df_prod[col] = LabelEncoder().fit_transform(df_prod[col].astype(str))

X_prod = df_prod.drop(['is_trending', 'click_through_rate', 'conversion_rate'], axis=1)

def train_product_task(y_series, task_name, objective='binary'):
    y = y_series
    X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
        X_prod, y, test_size=0.2,
        stratify=y if objective == 'binary' else None,
        random_state=RANDOM_SEED
    )
    cls = 'binary' if objective == 'binary' else 'regression'
    ModelClass = lgb.LGBMClassifier if objective == 'binary' else lgb.LGBMRegressor
    model = ModelClass(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        num_leaves=63, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
    )
    model.fit(X_tr_p, y_tr_p)
    y_pred = model.predict(X_te_p)
    if objective == 'binary':
        y_prob = model.predict_proba(X_te_p)[:, 1]
        metrics = eval_classifier(y_te_p, y_pred, y_prob, label=task_name)
    else:
        metrics = eval_regressor(y_te_p, y_pred, label=task_name)
    return model, metrics

trending_model, trending_metrics = train_product_task(df_prod['is_trending'], 'Trending XGBoost', 'binary')
ctr_model, ctr_metrics           = train_product_task(df_prod['click_through_rate'], 'CTR LightGBM', 'regression')
cvr_model, cvr_metrics           = train_product_task(df_prod['conversion_rate'], 'CVR LightGBM', 'regression')

plot_feature_importance(trending_model, X_prod.columns.tolist(), title='Trending — Feature Importance')

save_model_to_s3({'trending': trending_model, 'ctr': ctr_model, 'cvr': cvr_model},
                 'product-models',
                 metadata={'trending': trending_metrics, 'ctr': ctr_metrics, 'cvr': cvr_metrics})

## 9. Task 9: Search CTR Prediction (LightGBM)

In [ ]:
print('=' * 60)
print('TASK 9: Search CTR Prediction')
print('=' * 60)

search_feats = [
    'query_length', 'n_results', 'hour_of_day', 'day_of_week', 'is_weekend',
    'response_time_ms'
]
cat_search = ['sort_by', 'platform', 'filter_category']

df_srch = df_search[search_feats + cat_search + ['is_clicked', 'has_purchase']].copy()
df_srch['filter_category'] = df_srch['filter_category'].fillna('none')
df_srch = df_srch.dropna()

for col in cat_search:
    df_srch[col] = LabelEncoder().fit_transform(df_srch[col].astype(str))

X_srch = df_srch.drop(['is_clicked', 'has_purchase'], axis=1)
y_click = df_srch['is_clicked']

X_tr_sr, X_te_sr, y_tr_sr, y_te_sr = train_test_split(
    X_srch, y_click, test_size=0.2, stratify=y_click, random_state=RANDOM_SEED
)

search_ctr_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    num_leaves=127, class_weight='balanced',
    random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
)
search_ctr_model.fit(X_tr_sr, y_tr_sr)
y_pred_sr = search_ctr_model.predict(X_te_sr)
y_prob_sr = search_ctr_model.predict_proba(X_te_sr)[:, 1]
search_metrics = eval_classifier(y_te_sr, y_pred_sr, y_prob_sr, label='Search CTR LightGBM')

save_model_to_s3(search_ctr_model, 'search-ctr', metadata={'metrics': search_metrics})

## 10. Task 10: Collaborative Filtering (Matrix Factorization — PyTorch)

In [ ]:
print('=' * 60)
print('TASK 10: Collaborative Filtering — Neural Matrix Factorization')
print('=' * 60)

# ── Persiapan data ────────────────────────────────────────────────────────────
df_cf = df_interact[['user_id', 'product_id', 'implicit_score']].copy()

# Re-encode ID menjadi 0-based index
user_enc  = LabelEncoder().fit(df_cf['user_id'])
prod_enc  = LabelEncoder().fit(df_cf['product_id'])
df_cf['u'] = user_enc.transform(df_cf['user_id'])
df_cf['p'] = prod_enc.transform(df_cf['product_id'])

# Normalize score ke [0, 1]
max_score = df_cf['implicit_score'].max()
df_cf['score_norm'] = df_cf['implicit_score'] / max_score

n_users = df_cf['u'].nunique()
n_items = df_cf['p'].nunique()
print(f'Users: {n_users:,} | Items: {n_items:,} | Interactions: {len(df_cf):,}')

# Train/test split
cf_train, cf_test = train_test_split(df_cf, test_size=0.1, random_state=RANDOM_SEED)


# ── Dataset & DataLoader ──────────────────────────────────────────────────────
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.LongTensor(df['u'].values)
        self.items  = torch.LongTensor(df['p'].values)
        self.scores = torch.FloatTensor(df['score_norm'].values)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.scores[idx]


train_loader = DataLoader(InteractionDataset(cf_train), batch_size=4096, shuffle=True,  num_workers=4)
test_loader  = DataLoader(InteractionDataset(cf_test),  batch_size=4096, shuffle=False, num_workers=4)


# ── Neural MF Model ───────────────────────────────────────────────────────────
class NeuralMF(nn.Module):
    """Neural Matrix Factorization:
       GMF branch (element-wise product) + MLP branch.
    """
    def __init__(self, n_users, n_items, emb_dim=64, mlp_layers=[256, 128, 64], dropout=0.2):
        super().__init__()
        # GMF embeddings
        self.gmf_user = nn.Embedding(n_users, emb_dim)
        self.gmf_item = nn.Embedding(n_items, emb_dim)

        # MLP embeddings
        self.mlp_user = nn.Embedding(n_users, emb_dim)
        self.mlp_item = nn.Embedding(n_items, emb_dim)

        # MLP layers
        mlp = []
        in_size = emb_dim * 2
        for out_size in mlp_layers:
            mlp += [nn.Linear(in_size, out_size), nn.ReLU(), nn.Dropout(dropout)]
            in_size = out_size
        self.mlp = nn.Sequential(*mlp)

        # Output layer
        self.output = nn.Linear(emb_dim + mlp_layers[-1], 1)
        self.sigmoid = nn.Sigmoid()

        self._init_weights()

    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, user, item):
        # GMF branch
        gmf = self.gmf_user(user) * self.gmf_item(item)

        # MLP branch
        mlp_in = torch.cat([self.mlp_user(user), self.mlp_item(item)], dim=1)
        mlp_out = self.mlp(mlp_in)

        # Concat & predict
        out = self.output(torch.cat([gmf, mlp_out], dim=1))
        return self.sigmoid(out).squeeze()


# ── Training ──────────────────────────────────────────────────────────────────
cf_model = NeuralMF(n_users, n_items, emb_dim=64, mlp_layers=[256, 128, 64]).to(DEVICE)
optimizer = optim.Adam(cf_model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.MSELoss()

N_EPOCHS   = 10
best_val   = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    # Train
    cf_model.train()
    tr_loss = 0
    for u, p, s in train_loader:
        u, p, s = u.to(DEVICE), p.to(DEVICE), s.to(DEVICE)
        optimizer.zero_grad()
        pred = cf_model(u, p)
        loss = criterion(pred, s)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(u)
    tr_loss /= len(cf_train)

    # Validate
    cf_model.eval()
    val_loss = 0
    with torch.no_grad():
        for u, p, s in test_loader:
            u, p, s = u.to(DEVICE), p.to(DEVICE), s.to(DEVICE)
            pred = cf_model(u, p)
            val_loss += criterion(pred, s).item() * len(u)
    val_loss /= len(cf_test)

    scheduler.step(val_loss)
    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(cf_model.state_dict(), '/tmp/best_nmf.pt')

    print(f'  Epoch {epoch:02d}/{N_EPOCHS} — Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f}')

# Plot training curve
fig, ax = plt.subplots()
ax.plot(train_losses, label='Train MSE')
ax.plot(val_losses,   label='Val MSE')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Neural MF — Training Curve')
ax.legend(); plt.tight_layout(); plt.show()

print(f'\n✅ Best Validation MSE: {best_val:.4f}')

# Upload model ke S3
s3_client.upload_file('/tmp/best_nmf.pt', S3_BUCKET, 'models/collaborative-filtering/best_nmf.pt')
print('✅ NMF model saved to S3')

In [ ]:
# ── Inference: Top-K Recommendation ──────────────────────────────────────────
def recommend_top_k(user_raw_id: int, k: int = 10):
    """Generate top-K rekomendasi produk untuk seorang user."""
    cf_model.eval()
    try:
        u_idx = user_enc.transform([user_raw_id])[0]
    except ValueError:
        print(f'User {user_raw_id} tidak ditemukan.')
        return []

    # Produk yang sudah diinteraksi user ini
    seen = set(df_cf[df_cf['user_id'] == user_raw_id]['p'].values)
    all_items = [i for i in range(n_items) if i not in seen]

    with torch.no_grad():
        u_tensor = torch.LongTensor([u_idx] * len(all_items)).to(DEVICE)
        p_tensor = torch.LongTensor(all_items).to(DEVICE)
        scores   = cf_model(u_tensor, p_tensor).cpu().numpy()

    top_k_idx   = np.argsort(scores)[::-1][:k]
    top_k_items = prod_enc.inverse_transform([all_items[i] for i in top_k_idx])
    top_k_score = scores[top_k_idx]

    result = pd.DataFrame({'product_id': top_k_items, 'predicted_score': top_k_score})
    result = result.merge(df_products[['product_id', 'product_name', 'category', 'final_price_idr']],
                          on='product_id', how='left')
    return result

# Demo
sample_user = df_users['user_id'].iloc[0]
recs = recommend_top_k(sample_user, k=10)
print(f'\n🎯 Top-10 Rekomendasi untuk User {sample_user}:')
print(recs.to_string(index=False))

## 11. Task 11: Search Result Ranking (LightGBM LambdaRank)

In [ ]:
print('=' * 60)
print('TASK 11: Search Result Ranking — LightGBM LambdaRank')
print('=' * 60)

# Bangun dataset ranking:
# Query = kelompok, relevance = is_clicked + 2*has_purchase
df_rank = df_search.copy()
df_rank['filter_category'] = df_rank['filter_category'].fillna('none')
df_rank = df_rank.dropna(subset=['query', 'is_clicked', 'has_purchase'])

# Encode query sebagai group ID
le_q = LabelEncoder()
df_rank['query_id'] = le_q.fit_transform(df_rank['query'].astype(str))

# Relevance label (0–3 scale)
df_rank['relevance'] = df_rank['is_clicked'].astype(int) + 2 * df_rank['has_purchase'].astype(int)

rank_features = [
    'query_length', 'n_results', 'hour_of_day', 'day_of_week',
    'is_weekend', 'response_time_ms'
]
cat_rank = ['sort_by', 'platform', 'filter_category']
for col in cat_rank:
    df_rank[col] = LabelEncoder().fit_transform(df_rank[col].astype(str))

X_rank = df_rank[rank_features + cat_rank]
y_rank = df_rank['relevance']
g_rank = df_rank.groupby('query_id').size().values  # group sizes untuk ranking

# Split: ambil 80% query untuk train
unique_qids = df_rank['query_id'].unique()
train_qids  = np.random.choice(unique_qids, int(len(unique_qids) * 0.8), replace=False)
mask_train  = df_rank['query_id'].isin(train_qids)

X_tr_rk, X_te_rk = X_rank[mask_train], X_rank[~mask_train]
y_tr_rk, y_te_rk = y_rank[mask_train], y_rank[~mask_train]
g_tr_rk = df_rank[mask_train].groupby('query_id').size().values
g_te_rk = df_rank[~mask_train].groupby('query_id').size().values

# LightGBM LambdaRank
lgb_train_ds = lgb.Dataset(X_tr_rk, label=y_tr_rk, group=g_tr_rk)
lgb_test_ds  = lgb.Dataset(X_te_rk, label=y_te_rk, group=g_te_rk, reference=lgb_train_ds)

rank_params = {
    'objective':        'lambdarank',
    'metric':           'ndcg',
    'ndcg_eval_at':     [5, 10],
    'learning_rate':    0.05,
    'num_leaves':       63,
    'max_depth':        7,
    'min_child_samples': 50,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'label_gain':       [0, 1, 3],  # sesuai relevance 0,1,3
    'verbose':          -1,
    'seed':             RANDOM_SEED,
}

callbacks = [
    lgb.early_stopping(50),
    lgb.log_evaluation(100),
]

ranking_model = lgb.train(
    rank_params,
    lgb_train_ds,
    num_boost_round=1000,
    valid_sets=[lgb_test_ds],
    callbacks=callbacks,
)

# Evaluasi NDCG@10
preds_rk = ranking_model.predict(X_te_rk)
print(f'\n✅ Best NDCG@5  : {ranking_model.best_score["valid_0"]["ndcg@5"]:.4f}')
print(f'   Best NDCG@10 : {ranking_model.best_score["valid_0"]["ndcg@10"]:.4f}')

# Feature importance ranking model
fi = pd.Series(ranking_model.feature_importance(), index=rank_features + cat_rank).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
fi.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Search Ranking — Feature Importance (LambdaRank)')
ax.set_xlabel('Feature'); ax.set_ylabel('Importance')
plt.tight_layout(); plt.show()

save_model_to_s3(ranking_model, 'search-ranking',
    metadata={'ndcg@5': ranking_model.best_score['valid_0']['ndcg@5'],
              'ndcg@10': ranking_model.best_score['valid_0']['ndcg@10']})

## 12. Model Registry & Summary

In [ ]:
print('=' * 70)
print('📋 TECHNOMART ML MODEL REGISTRY — Training Summary')
print('=' * 70)

registry = [
    {'Task': 'Churn Prediction',    'Model': 'XGBoost',           'Key Metric': f'AUC = {churn_metrics.get("auc", 0):.4f}',   'S3 Path': 'models/churn-prediction/'},
    {'Task': 'User Segmentation',   'Model': 'XGBoost Multi',     'Key Metric': f'F1  = {seg_metrics.get("f1_weighted", 0):.4f}',   'S3 Path': 'models/user-segmentation/'},
    {'Task': 'LTV Prediction',      'Model': 'LightGBM Regr.',    'Key Metric': f'R²  = {ltv_metrics.get("r2", 0):.4f}',      'S3 Path': 'models/ltv-prediction/'},
    {'Task': 'Fraud Detection',     'Model': 'XGBoost + SMOTE',   'Key Metric': f'AUC = {fraud_metrics.get("auc", 0):.4f}',   'S3 Path': 'models/fraud-detection/'},
    {'Task': 'Sentiment Analysis',  'Model': 'LightGBM + TF-IDF', 'Key Metric': f'F1  = {sent_metrics.get("f1_weighted", 0):.4f}',  'S3 Path': 'models/review-models/'},
    {'Task': 'Rating Prediction',   'Model': 'LightGBM Regr.',    'Key Metric': f'R²  = {rating_metrics.get("r2", 0):.4f}',   'S3 Path': 'models/review-models/'},
    {'Task': 'Trending Detection',  'Model': 'LightGBM',          'Key Metric': f'AUC = {trending_metrics.get("auc", 0):.4f}','S3 Path': 'models/product-models/'},
    {'Task': 'CTR Prediction',      'Model': 'LightGBM Regr.',    'Key Metric': f'R²  = {ctr_metrics.get("r2", 0):.4f}',      'S3 Path': 'models/product-models/'},
    {'Task': 'CVR Prediction',      'Model': 'LightGBM Regr.',    'Key Metric': f'R²  = {cvr_metrics.get("r2", 0):.4f}',      'S3 Path': 'models/product-models/'},
    {'Task': 'Search CTR',          'Model': 'LightGBM',          'Key Metric': f'AUC = {search_metrics.get("auc", 0):.4f}',  'S3 Path': 'models/search-ctr/'},
    {'Task': 'Collaborative Filter','Model': 'Neural MF (PyTorch)','Key Metric': f'MSE = {best_val:.4f}',                      'S3 Path': 'models/collaborative-filtering/'},
    {'Task': 'Search Ranking',      'Model': 'LightGBM LambdaRank','Key Metric': f'NDCG@10 = {ranking_model.best_score["valid_0"]["ndcg@10"]:.4f}', 'S3 Path': 'models/search-ranking/'},
]

df_registry = pd.DataFrame(registry)
print(df_registry.to_string(index=False))

# Save registry ke S3
buf = StringIO()
df_registry.to_csv(buf, index=False)
s3_client.put_object(
    Bucket=S3_BUCKET,
    Key='models/model_registry.csv',
    Body=buf.getvalue()
)
print(f'\n✅ Model registry saved: s3://{S3_BUCKET}/models/model_registry.csv')
print('\n🎉 Semua training selesai!')